# Análise do OSM

Escolas e unidades de saúde de Porto Alegre coletadas do OpenStreetMap (Overpass API).

In [1]:
import json
from pathlib import Path

import pandas as pd

# Pasta raiz do repositório (funciona rodando de notebooks/ ou da raiz)
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

## Tabela de escolas e unidades de saúde

In [2]:
with open(RAIZ / "dados" / "originais" / "osm" / "osm_escolas_saude_poa.json", encoding="utf-8") as f:
    osm = pd.json_normalize(json.load(f)["elements"])

# Ways/relations não têm lat/lon próprios: usa o centro calculado pelo Overpass
osm["lat"] = osm["lat"].fillna(osm["center.lat"])
osm["lon"] = osm["lon"].fillna(osm["center.lon"])

colunas = ["type", "id", "tags.amenity", "tags.name", "tags.addr:street",
           "tags.addr:suburb", "lat", "lon"]
colunas = [c for c in colunas if c in osm.columns]
print(f"{len(osm)} pontos")
display(osm["tags.amenity"].value_counts().to_frame("quantidade"))
osm[colunas]

708 pontos


,quantidade
tags.amenity,
school,438
clinic,98
kindergarten,87
hospital,52
doctors,33


,type,id,tags.amenity,tags.name,tags.addr:street,tags.addr:suburb,lat,lon
0,node,476647429,hospital,Unidade Álvaro Alvim - HCPA,Rua Professor Álvaro Alvim,Rio Branco,-30.037137,-51.198245
1,node,598724565,clinic,Estratégia Saúde da Família Tijuca,NaN,NaN,-30.042889,-51.126231
2,node,1118102920,school,Colégio Marista Ipanema,Coronel Marcos,NaN,-30.130579,-51.239623
3,node,1139635710,kindergarten,Escola Moinho,Avenida Soledade,NaN,-30.034895,-51.180255
4,node,1139971906,kindergarten,Creche Berçário e recreação Baby House,Avenida Doutor Nilo Peçanha,NaN,-30.029053,-51.178558
...,...,...,...,...,...,...,...,...
703,relation,6344229,school,Escola Cristóvão Colombo,Rua Rezende Costa,NaN,-29.983076,-51.131544
704,relation,6346576,clinic,Posto de Saúde,Beco José Paris,Sarandi,-29.988678,-51.116854
705,relation,6354469,clinic,Unidade de Saúde SESC,Rua Ernesto Pellanda,Bairro Vila Jardim,-30.035936,-51.148589
706,relation,8397517,school,Colégio Leonardo da Vinci - Unidade Beta,Avenida Icaraí,Cristal,-30.094842,-51.245732


## Mapa

In [3]:
import plotly.express as px

px.scatter_map(osm, lat="lat", lon="lon", color="tags.amenity", hover_name="tags.name",
               zoom=10, height=600, map_style="carto-positron")

## Pontos com nome repetido

In [4]:
# Compara nomes sem diferenciar maiúsculas/minúsculas nem espaços extras
osm["nome_norm"] = osm["tags.name"].str.strip().str.lower()
repetidos = (osm[osm["nome_norm"].notna() & osm.duplicated("nome_norm", keep=False)]
             .sort_values(["nome_norm", "tags.amenity"]))
print(f"{repetidos['nome_norm'].nunique()} nomes repetidos em {len(repetidos)} pontos")
with pd.option_context("display.max_rows", None):
    display(repetidos[colunas])

12 nomes repetidos em 27 pontos


,type,id,tags.amenity,tags.name,tags.addr:street,tags.addr:suburb,lat,lon
472,way,419339793,clinic,Centro Clínico Gaúcho,Avenida Alberto Bins,NaN,-30.028066,-51.218551
224,way,239664213,hospital,Centro Clínico Gaúcho,Avenida do Forte,NaN,-30.012187,-51.153793
123,node,11030466129,school,Colégio Sinodal do Salvador,NaN,NaN,-30.013463,-51.147306
446,way,367324147,school,Colégio Sinodal do Salvador,Avenida Mathilde Trein Renner,NaN,-30.013599,-51.147116
513,way,425196135,kindergarten,Creche Comunitária Tia Ivete,NaN,NaN,-30.048789,-51.150592
514,way,425196740,kindergarten,Creche Comunitária Tia Ivete,NaN,NaN,-30.045210,-51.148295
204,way,235379350,school,Escola Afonso Guerreiro Lima,Rua Banco Inglês,NaN,-30.071847,-51.221474
236,way,243118489,school,Escola Afonso Guerreiro Lima,Rua Guaíba,NaN,-30.110979,-51.112808
276,way,244250676,school,Escola América,Rua Octacílio José Dias,NaN,-30.015383,-51.121649
378,way,317875313,school,Escola América,Rua Padre Ângelo Costa,NaN,-30.078842,-51.167755


In [5]:
# Mapa só com os pontos de nome repetido (cor = nome, símbolo = tipo OSM)
px.scatter_map(repetidos, lat="lat", lon="lon", color="tags.name",
               hover_name="tags.name", hover_data=["type", "id", "tags.amenity"],
               zoom=10, height=600, map_style="carto-positron")

## Análise de duplicatas

Após filtrar os nomes repetidos, exibimos estes pontos no mapa cima e analisamos manualmente quais pontos representam lugares diferentes e quais são redundâncias de um mesmo local.

Duplicatas:
- Unimed: ID's 4117912914 e 420868536
- Escola Visconde do Rio Grande: ID 243463347	
- Escola Saint Hilaire: ID 428647547
- Escola Estadual de 1° Grau Professora Dinah Néri Pereira: ID 6238095094
- Colégio Sinodal do Salvador: ID 11030466129

## Pontos sem nome

In [6]:
sem_nome = osm[osm["tags.name"].isna()]
print(f"{len(sem_nome)} pontos sem nome")
display(sem_nome["tags.amenity"].value_counts().to_frame("quantidade"))
with pd.option_context("display.max_rows", None):
    display(sem_nome[colunas])

24 pontos sem nome


,quantidade
tags.amenity,
school,12
clinic,5
kindergarten,3
hospital,3
doctors,1


,type,id,tags.amenity,tags.name,tags.addr:street,tags.addr:suburb,lat,lon
34,node,3428745569,kindergarten,NaN,NaN,NaN,-30.033714,-51.234507
119,node,8730543929,clinic,NaN,NaN,NaN,-30.043274,-51.153660
120,node,8730543930,clinic,NaN,NaN,NaN,-30.043276,-51.153849
218,way,237709752,hospital,NaN,NaN,NaN,-30.015223,-51.158118
275,way,244050597,school,NaN,NaN,NaN,-30.025459,-51.117782
304,way,249901840,clinic,NaN,NaN,NaN,-30.113241,-51.165271
334,way,288987786,school,NaN,NaN,NaN,-29.992983,-51.192296
335,way,288987796,doctors,NaN,NaN,NaN,-29.992663,-51.193657
372,way,317192910,school,NaN,NaN,NaN,-30.035335,-51.233530
388,way,318661243,hospital,NaN,NaN,NaN,-30.106660,-51.171792


### Pontos sem nome a menos de 200 m um do outro (possíveis duplicatas)

In [7]:
import numpy as np

LIMITE_M = 200

def haversine_m(lat1, lon1, lat2, lon2):
    """Distância em metros entre coordenadas (lat/lon em graus)."""
    R = 6_371_000
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = (np.sin((lat2 - lat1) / 2) ** 2
         + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2)
    return 2 * R * np.arcsin(np.sqrt(a))

s = sem_nome.reset_index(drop=True)
# Matriz de distâncias entre todos os pares de pontos sem nome
dist = haversine_m(s["lat"].values[:, None], s["lon"].values[:, None],
                   s["lat"].values[None, :], s["lon"].values[None, :])
i, j = np.where(np.triu(dist < LIMITE_M, k=1))  # cada par uma vez só

pares = pd.DataFrame({
    "id_1": s.loc[i, "id"].values, "tipo_1": s.loc[i, "type"].values,
    "amenity_1": s.loc[i, "tags.amenity"].values,
    "id_2": s.loc[j, "id"].values, "tipo_2": s.loc[j, "type"].values,
    "amenity_2": s.loc[j, "tags.amenity"].values,
    "distancia_m": dist[i, j].round(1),
}).sort_values("distancia_m")
print(f"{len(pares)} pares de pontos sem nome a menos de {LIMITE_M} m")
pares

12 pares de pontos sem nome a menos de 200 m


,id_1,tipo_1,amenity_1,id_2,tipo_2,amenity_2,distancia_m
11,425216986,way,school,425217265,way,school,10.7
6,425215193,way,school,425215214,way,school,11.5
0,8730543929,node,clinic,8730543930,node,clinic,18.2
8,425215193,way,school,425217265,way,school,24.2
10,425215214,way,school,425217265,way,school,27.6
7,425215193,way,school,425216986,way,school,34.5
4,425215043,way,school,425216986,way,school,35.1
5,425215043,way,school,425217265,way,school,36.5
9,425215214,way,school,425216986,way,school,38.2
3,425215043,way,school,425215214,way,school,48.7


# Limpeza dos dados

Tratamentos decididos na análise acima. Cada alteração está comentada no código com **ALTERAÇÃO**.

| Problema encontrado | Tratamento |
|---|---|
| Ways e relations sem `lat`/`lon` próprios | usar o centro calculado pelo Overpass (`center.lat`/`center.lon`) |
| Pontos com nome repetido que são o mesmo lugar (análise manual) | remover os IDs listados em "Análise de duplicatas" |
| Pontos sem nome, da mesma categoria, a menos de 200 m um do outro | manter só um ponto de cada grupo |
| Muitas colunas de tags que não usamos | manter só id, tipo, categoria, nome, endereço e coordenadas |
| Pontos sem nome que não são duplicatas | **manter** (são escolas/unidades reais, só não têm nome no OSM) |

In [8]:
osm_limpo = osm.copy()

# ALTERAÇÃO: ways/relations não têm lat/lon; usa o centro do polígono calculado pelo Overpass
osm_limpo["lat"] = osm_limpo["lat"].fillna(osm_limpo["center.lat"])
osm_limpo["lon"] = osm_limpo["lon"].fillna(osm_limpo["center.lon"])
assert osm_limpo[["lat", "lon"]].notna().all().all()
assert osm_limpo["id"].is_unique
print("Pontos antes da limpeza:", len(osm_limpo))

Pontos antes da limpeza: 708


## Remoção das duplicatas com nome (análise manual)

IDs identificados no mapa de nomes repetidos como o mesmo lugar mapeado mais de uma vez.

In [9]:
# ALTERAÇÃO: remove os pontos duplicados encontrados na análise manual
IDS_DUPLICADOS = {
    4117912914: "Unimed",
    420868536: "Unimed",
    243463347: "Escola Visconde do Rio Grande",
    428647547: "Escola Saint Hilaire",
    6238095094: "Escola Estadual de 1° Grau Professora Dinah Néri Pereira",
    11030466129: "Colégio Sinodal do Salvador",
}

removidos_manual = osm_limpo[osm_limpo["id"].isin(IDS_DUPLICADOS)]
# Confere que todos os IDs existem e que o nome bate com o esperado
assert len(removidos_manual) == len(IDS_DUPLICADOS), "algum ID não foi encontrado"
assert (removidos_manual["tags.name"] == removidos_manual["id"].map(IDS_DUPLICADOS)).all()

osm_limpo = osm_limpo[~osm_limpo["id"].isin(IDS_DUPLICADOS)]
print(f"Removidos {len(removidos_manual)} pontos | restam {len(osm_limpo)}")
removidos_manual[colunas]

Removidos 6 pontos | restam 702


,type,id,tags.amenity,tags.name,tags.addr:street,tags.addr:suburb,lat,lon
47,node,4117912914,clinic,Unimed,NaN,NaN,-30.010201,-51.148266
87,node,6238095094,school,Escola Estadual de 1° Grau Professora Dinah Né...,NaN,NaN,-30.037782,-51.212529
123,node,11030466129,school,Colégio Sinodal do Salvador,NaN,NaN,-30.013463,-51.147306
252,way,243463347,school,Escola Visconde do Rio Grande,Praça Aratiba,NaN,-30.102466,-51.229183
491,way,420868536,clinic,Unimed,NaN,NaN,-30.023903,-51.200695
586,way,428647547,school,Escola Saint Hilaire,Rua Gervazio Braga Pinheiro,NaN,-30.123816,-51.097750


## Remoção das duplicatas sem nome (proximidade)

Usa os pares da última célula da análise (`pares`: pontos sem nome a menos de 200 m).
Regras:
* só são duplicatas pontos da **mesma categoria** (`amenity`) — uma escola ao lado de um consultório são lugares diferentes;
* pares encadeados formam um grupo (A perto de B e B perto de C → A, B e C são o mesmo lugar);
* de cada grupo **mantemos o ponto de menor ID** (o mapeado primeiro no OSM) e removemos os outros.

In [10]:
# Só pares da mesma categoria
mesma_cat = pares[pares["amenity_1"] == pares["amenity_2"]]

# Agrupa pares encadeados: cada ponto aponta para o menor ID do seu grupo
grupo = {}
def raiz(x):
    while grupo.get(x, x) != x:
        x = grupo[x]
    return x
for a, b in zip(mesma_cat["id_1"], mesma_cat["id_2"]):
    ra, rb = raiz(a), raiz(b)
    grupo[max(ra, rb)] = min(ra, rb)

ids_no_grupo = set(mesma_cat["id_1"]) | set(mesma_cat["id_2"])
grupos = pd.DataFrame({"id": sorted(ids_no_grupo)})
grupos["grupo"] = grupos["id"].map(raiz)
grupos["acao"] = np.where(grupos["id"] == grupos["grupo"], "manter", "remover")
grupos = grupos.merge(osm[["id", "type", "tags.amenity", "lat", "lon"]], on="id")
display(grupos)

# ALTERAÇÃO: remove os pontos sem nome duplicados (mantém um por grupo)
ids_remover = grupos.loc[grupos["acao"] == "remover", "id"]
osm_limpo = osm_limpo[~osm_limpo["id"].isin(ids_remover)]
print(f"Removidos {len(ids_remover)} pontos sem nome | restam {len(osm_limpo)}")

,id,grupo,acao,type,tags.amenity,lat,lon
0,425215043,425215043,manter,way,school,-30.051108,-51.163121
1,425215193,425215043,remover,way,school,-30.051402,-51.162681
2,425215214,425215043,remover,way,school,-30.051441,-51.162791
3,425216986,425215043,remover,way,school,-30.051098,-51.162756
4,425217265,425215043,remover,way,school,-30.051195,-51.162755
5,8730543929,8730543929,manter,node,clinic,-30.043274,-51.153660
6,8730543930,8730543929,remover,node,clinic,-30.043276,-51.153849


Removidos 5 pontos sem nome | restam 697


## Seleção de colunas e verificação

In [11]:
# ALTERAÇÃO: mantém só as colunas usadas e renomeia para nomes simples
osm_final = osm_limpo.rename(columns={
    "id": "id_osm", "type": "tipo_osm", "tags.amenity": "categoria", "tags.name": "nome",
    "tags.addr:street": "rua", "tags.addr:housenumber": "numero", "tags.addr:suburb": "bairro",
})[["id_osm", "tipo_osm", "categoria", "nome", "rua", "numero", "bairro", "lat", "lon"]]
osm_final = osm_final.sort_values(["categoria", "nome"]).reset_index(drop=True)

assert osm_final["id_osm"].is_unique
assert osm_final[["lat", "lon"]].notna().all().all()
assert not osm_final["id_osm"].isin(IDS_DUPLICADOS).any()

print(f"Antes: {len(osm)} pontos | depois: {len(osm_final)} pontos | removidos: {len(osm) - len(osm_final)}")
pd.DataFrame({"antes": osm["tags.amenity"].value_counts(),
              "depois": osm_final["categoria"].value_counts()})

Antes: 708 pontos | depois: 697 pontos | removidos: 11


,antes,depois
school,438,430
clinic,98,95
kindergarten,87,87
hospital,52,52
doctors,33,33


## Exportação

Formato longo: uma linha por estabelecimento, salvo como lista de objetos JSON em `dados/processados/osm/`.
Campos vazios no OSM (nome, endereço) ficam como `null`.

In [12]:
SAIDA = RAIZ / "dados" / "processados" / "osm"
SAIDA.mkdir(parents=True, exist_ok=True)
arquivo = SAIDA / "osm_escolas_saude_poa.json"

registros = osm_final.astype(object).where(osm_final.notna(), None)  # NaN -> null no JSON
with open(arquivo, "w", encoding="utf-8") as f:
    json.dump(registros.to_dict(orient="records"), f, ensure_ascii=False, indent=1)

print("Salvo:", arquivo, "|", len(registros), "registros")
pd.read_json(arquivo).head()

Salvo: /home/piedrorn/Documentos/puc/coleta/projeto-enchentes/dados/processados/osm/osm_escolas_saude_poa.json | 697 registros


,id_osm,tipo_osm,categoria,nome,rua,numero,bairro,lat,lon
0,414432876,way,clinic,AmorSaúde,Avenida Alberto Bins,391,Centro Histórico,-30.028339,-51.222780
1,7180073227,node,clinic,Andrea Mendonça Rodrigues - Pneumologista Pedi...,Rua General João Telles,542-601,NaN,-30.034159,-51.212849
2,4763286051,node,clinic,Banco de Sangue do Hospital de Clínicas,Rua São Manoel,543,NaN,-30.038756,-51.205235
3,485713863,way,clinic,CAF Fisioterapia,Rua Vasco da Gama,485,NaN,-30.032264,-51.207225
4,6805506414,node,clinic,CAPS II HCPA,Rua Ramiro Barcelos,2350,Santa Cecília,-30.038660,-51.207129
